# 03 — Gold Aggregate: SaaS Product & Customer Analytics
Builds six analytical Gold tables from the Silver layer.

| Gold Table | Description |
|---|---|
| gold_account_health | Per-account MRR, health score, churn risk |
| gold_feature_adoption | Feature usage counts, unique users, avg duration |
| gold_churn_analysis | Churned accounts by plan, region, industry |
| gold_support_performance | SLA breach rates, avg CSAT, by priority |
| gold_weekly_engagement_trends | Weekly DAU proxy and event volume |
| gold_account_scorecards | Multi-signal account health scorecard |

In [ ]:
LAKEHOUSE_NAME = "technology_lakehouse"
SILVER = "silver"
GOLD   = "gold"

from pyspark.sql import functions as F

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {LAKEHOUSE_NAME}.{GOLD}")

sa  = spark.table(f"{LAKEHOUSE_NAME}.{SILVER}.silver_accounts")
su  = spark.table(f"{LAKEHOUSE_NAME}.{SILVER}.silver_users")
sev = spark.table(f"{LAKEHOUSE_NAME}.{SILVER}.silver_events")
stk = spark.table(f"{LAKEHOUSE_NAME}.{SILVER}.silver_support_tickets")

In [ ]:
# ── gold_account_health ──────────────────────────────────────────────────────
user_activity = (
    su.groupBy("account_id")
    .agg(
        F.count("user_id").alias("total_users"),
        F.sum("is_active").alias("active_users"),
        F.round(F.avg("logins_last_30_days"), 1).alias("avg_logins_30d"),
    )
)

gold_ah = (
    sa
    .join(user_activity, on="account_id", how="left")
    .select(
        "account_id", "plan", "mrr_usd", "industry", "region",
        "health_score", "is_churned", "churn_risk_flag",
        "days_as_customer", "seat_count", "plan_tier",
        "total_users", "active_users", "avg_logins_30d"
    )
    .withColumn("_updated_at", F.current_timestamp())
)
gold_ah.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable(f"{LAKEHOUSE_NAME}.{GOLD}.gold_account_health")
print(f"gold_account_health: {gold_ah.count():,}")

In [ ]:
# ── gold_feature_adoption ────────────────────────────────────────────────────
gold_fa = (
    sev
    .groupBy("feature", "feature_category")
    .agg(
        F.count("event_id").alias("total_events"),
        F.countDistinct("user_id").alias("unique_users"),
        F.countDistinct("account_id").alias("unique_accounts"),
        F.round(F.avg("duration_secs"), 1).alias("avg_duration_secs"),
    )
    .withColumn("_updated_at", F.current_timestamp())
)
gold_fa.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable(f"{LAKEHOUSE_NAME}.{GOLD}.gold_feature_adoption")
print(f"gold_feature_adoption: {gold_fa.count():,}")

In [ ]:
# ── gold_churn_analysis ──────────────────────────────────────────────────────
gold_ca = (
    sa.filter(F.col("is_churned") == 1)
    .groupBy("plan", "region", "industry")
    .agg(
        F.count("account_id").alias("churned_accounts"),
        F.round(F.sum("mrr_usd"), 0).alias("lost_mrr_usd"),
        F.round(F.avg("days_as_customer"), 0).alias("avg_tenure_days"),
        F.round(F.avg("health_score"), 1).alias("avg_health_at_churn"),
    )
    .withColumn("_updated_at", F.current_timestamp())
)
gold_ca.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable(f"{LAKEHOUSE_NAME}.{GOLD}.gold_churn_analysis")
print(f"gold_churn_analysis: {gold_ca.count():,}")

In [ ]:
# ── gold_support_performance ─────────────────────────────────────────────────
gold_sp = (
    stk
    .groupBy("priority", "category")
    .agg(
        F.count("ticket_id").alias("total_tickets"),
        F.sum("is_sla_breached").alias("sla_breaches"),
        F.round(F.avg("resolution_hrs"), 2).alias("avg_resolution_hrs"),
        F.round(F.avg("csat_score"), 2).alias("avg_csat"),
    )
    .withColumn("sla_breach_rate_pct",
        F.round(F.col("sla_breaches") / F.col("total_tickets") * 100, 1))
    .withColumn("_updated_at", F.current_timestamp())
)
gold_sp.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable(f"{LAKEHOUSE_NAME}.{GOLD}.gold_support_performance")
print(f"gold_support_performance: {gold_sp.count():,}")

In [ ]:
# ── gold_weekly_engagement_trends ────────────────────────────────────────────
gold_wet = (
    sev
    .groupBy("event_week")
    .agg(
        F.count("event_id").alias("total_events"),
        F.countDistinct("user_id").alias("active_users_dau_proxy"),
        F.countDistinct("account_id").alias("active_accounts"),
        F.round(F.avg("duration_secs"), 1).alias("avg_session_secs"),
    )
    .orderBy("event_week")
    .withColumn("_updated_at", F.current_timestamp())
)
gold_wet.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable(f"{LAKEHOUSE_NAME}.{GOLD}.gold_weekly_engagement_trends")
print(f"gold_weekly_engagement_trends: {gold_wet.count():,}")

In [ ]:
# ── gold_account_scorecards ──────────────────────────────────────────────────
ticket_by_acc = stk.groupBy("account_id").agg(
    F.count("ticket_id").alias("total_tickets"),
    F.sum("is_sla_breached").alias("sla_breaches"),
    F.round(F.avg("csat_score"), 2).alias("avg_csat"),
)

event_by_acc = sev.groupBy("account_id").agg(
    F.count("event_id").alias("total_events"),
    F.countDistinct("feature").alias("features_used"),
)

gold_asc = (
    sa.select("account_id", "plan", "mrr_usd", "health_score", "is_churned", "churn_risk_flag", "region", "industry")
    .join(ticket_by_acc, on="account_id", how="left")
    .join(event_by_acc,  on="account_id", how="left")
    .withColumn("_updated_at", F.current_timestamp())
)
gold_asc.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable(f"{LAKEHOUSE_NAME}.{GOLD}.gold_account_scorecards")
print(f"gold_account_scorecards: {gold_asc.count():,}")

In [ ]:
print("\n=== Gold Aggregate Complete ===")
gold_tables = [
    "gold_account_health", "gold_feature_adoption", "gold_churn_analysis",
    "gold_support_performance", "gold_weekly_engagement_trends", "gold_account_scorecards"
]
for tbl in gold_tables:
    cnt = spark.table(f"{LAKEHOUSE_NAME}.{GOLD}.{tbl}").count()
    print(f"  {tbl}: {cnt:,}")